# RadioSkySim — Waterfall

Este notebook demonstra as principais funcionalidades públicas do pacote
`radioskysims-waterfall` usando um exemplo **leve, determinístico e independente
de banco de dados, Space-Track ou rede**.

O objetivo é partir de um dado sintético do tipo **tempo × frequência** e
percorrer o fluxo:

**TOD 2D → Waterfall → processamento → SED → LightCurve → visualização**

Serão demonstrados:

- criação de `Waterfall`;
- metadados;
- seleção de dados;
- filtros espectral e temporal;
- estimativa e remoção de baseline;
- normalização;
- API chainable;
- `SED`;
- `LightCurve`;
- `WaterfallPipeline`;
- comparação visual entre dados originais e processados.


## 1. Imports e configuração

Utilizamos apenas a API pública de alto nível do pacote Waterfall, além de
NumPy e xarray para construir um TOD sintético de demonstração.


In [ ]:
# ── Âncora de diretório: raiz do projeto (CWD-independente) ──
# Garante que caminhos relativos (ex.: data/TOD/...) resolvam da raiz do
# projeto, independente de onde o kernel Jupyter iniciou (notebooks/ ou raiz).
import os

from radioskysims_core.paths import get_project_root

os.chdir(get_project_root())
print("cwd →", os.getcwd())

In [ ]:
from __future__ import annotations

import numpy as np
import xarray as xr

from radioskysims_waterfall import (
    LightCurve,
    SED,
    Waterfall,
    WaterfallMetadata,
    WaterfallPipeline,
)

print("API pública Waterfall carregada com sucesso.")


## 2. Construção de um TOD sintético

Criamos uma grade pequena de tempo × frequência contendo:

1. um nível de fundo suave;
2. uma dependência espectral;
3. ruído aleatório reproduzível;
4. uma faixa espectral mais intensa;
5. um evento localizado no tempo e na frequência.

O exemplo é propositalmente pequeno para ser executado rapidamente.


In [ ]:
rng = np.random.default_rng(42)

N_TIME = 120
N_FREQ = 96

time = np.datetime64("2026-09-08T00:00:00") + np.arange(N_TIME) * np.timedelta64(5, "s")

freq = np.linspace(980.0e6, 1260.0e6, N_FREQ)

# Fundo suave em tempo e frequência.
time_axis = np.linspace(0.0, 4.0 * np.pi, N_TIME)[:, None]
freq_axis = np.linspace(-1.0, 1.0, N_FREQ)[None, :]

background = 2.5 + 0.15 * np.sin(time_axis)
spectral_shape = 0.30 * freq_axis + 0.12 * freq_axis**2

# Ruído determinístico.
noise = rng.normal(0.0, 0.04, size=(N_TIME, N_FREQ))

values = background + spectral_shape + noise

# Faixa espectral persistente artificial.
values[:, 62:67] += 0.35

# Evento localizado artificial.
values[48:64, 38:48] += 1.20

tod = xr.DataArray(
    values,
    dims=("time", "frequency"),
    coords={
        "time": time,
        "frequency": freq,
    },
    name="T_antenna",
    attrs={"unit": "K"},
)

print("TOD sintético:", tod.shape)
print(
    "Faixa:",
    round(float(tod.min()), 3),
    "—",
    round(float(tod.max()), 3),
    "K",
)


## 3. Criação do `Waterfall`

O objeto `Waterfall` normaliza internamente as dimensões para a representação
canônica `("time", "freq")` e associa os metadados observacionais.


In [ ]:
wf = Waterfall(
    tod,
    time_name="time",
    freq_name="frequency",
    meta=WaterfallMetadata(
        name="tod_sintetico_demo",
        instrument="RadioSkySim",
        band="980-1260 MHz",
        polarization="I",
        data_unit="K",
        freq_unit="Hz",
    ),
)

print(wf)
print("shape:", wf.shape)
print("n_time:", wf.n_time)
print("n_freq:", wf.n_freq)
print("lazy:", wf.is_lazy)
print("memória estimada:", wf.estimate_memory(), "bytes")
print("metadata:", wf.meta)


## 4. Visualização do Waterfall original

O método público `Waterfall.plot()` produz a visualização tempo × frequência
e pode acrescentar automaticamente os painéis de curva de luz e SED.


In [ ]:
wf.plot(
    title="Waterfall sintético — dados originais",
    use_datashader=False,
    width=480,
    height=280,
)


## 5. Seleção de uma região

As operações do `Waterfall` são **chainable** e não modificam o objeto
original. O método `slice()` permite selecionar regiões por índices.


In [ ]:
wf_region = wf.slice(
    time=slice(35, 80),
    freq=slice(25, 75),
)

print("original:", wf.shape)
print("região:", wf_region.shape)
print("objeto original preservado:", wf.shape == (N_TIME, N_FREQ))


## 6. Filtros espectral e temporal

O filtro espectral atua ao longo dos canais de frequência para cada instante.
O filtro temporal atua ao longo do tempo para cada canal.

Aqui os mostramos separadamente para tornar o efeito de cada operação claro.


In [ ]:
wf_spectral = wf.filter_spectral(
    method="median",
    window=5,
    mode="subtract",
)

wf_temporal = wf.filter_temporal(
    method="median",
    window=7,
    mode="subtract",
)

print("spectral:", wf_spectral)
print("temporal:", wf_temporal)


## 7. Baseline e normalização

O pacote permite estimar o baseline e subtraí-lo diretamente. Também oferece
diversos esquemas de normalização. Neste exemplo usamos `minmax`, que leva os
dados para o intervalo aproximado `[0, 1]`.


In [ ]:
baseline = wf.get_baseline(
    method="median",
    window=9,
)

wf_no_baseline = wf.subtract_baseline(
    method="median",
    window=9,
)

wf_normalized = wf_no_baseline.normalize(method="minmax")

print("baseline:", baseline.shape)
print(
    "normalizado:",
    round(float(wf_normalized.data.min()), 4),
    "—",
    round(float(wf_normalized.data.max()), 4),
)


## 8. API chainable

Como cada método devolve um novo `Waterfall`, operações podem ser encadeadas
diretamente.


In [ ]:
wf_chain = (
    wf.filter_spectral(method="median", window=5, mode="subtract")
    .filter_temporal(method="median", window=7, mode="subtract")
    .normalize(method="minmax")
)

print(wf_chain)
print(
    "faixa processada:",
    round(float(wf_chain.data.min()), 4),
    "—",
    round(float(wf_chain.data.max()), 4),
)


## 9. SED — distribuição espectral média

A API pública `Waterfall.sed()` calcula a **Spectral Energy Distribution
(SED)**, isto é, o perfil médio do sinal em função da frequência ao longo do
intervalo temporal selecionado.

O objeto retornado também fornece média, mediana, desvio padrão e número de
amostras utilizadas.


In [ ]:
sed: SED = wf_chain.sed(name="SED processada")

print(sed)
print("canais:", len(sed))
print("amostras temporais:", sed.n_samples)
print("shape:", sed.shape)

sed.plot(
    title="SED — sinal médio por canal",
    width=650,
    height=300,
)


## 10. Light Curve — evolução temporal

A API pública `Waterfall.lightcurve()` calcula a curva de luz como a média do
sinal sobre uma faixa de frequências para cada instante de tempo.


In [ ]:
lightcurve: LightCurve = wf_chain.lightcurve(
    frequencies=slice(30, 60),
    name="Light Curve processada",
)

print(lightcurve)
print("pontos temporais:", len(lightcurve))
print("canais utilizados:", lightcurve.n_channels)

lightcurve.plot(
    title="Light Curve — evolução temporal",
    width=650,
    height=300,
)


## 11. Pipeline declarativa

Além do encadeamento direto, o pacote fornece `WaterfallPipeline`.

A pipeline armazena uma sequência explícita de operações, o que facilita:

- reproduzir um fluxo de processamento;
- inspecionar as etapas configuradas;
- reutilizar a mesma sequência em diferentes waterfalls;
- futuramente integrar controle de recursos por `SystemMonitor`.


In [ ]:
pipeline = (
    WaterfallPipeline()
    .add(
        "filter_spectral",
        method="median",
        window=5,
        mode="subtract",
    )
    .add(
        "filter_temporal",
        method="median",
        window=7,
        mode="subtract",
    )
    .add(
        "normalize",
        method="minmax",
    )
)

print("Pipeline:")
for step in pipeline.describe():
    print(" ", step)

wf_pipeline = pipeline.run(wf)

print()
print("Resultado:", wf_pipeline)


## 12. Comparação original × processado

O próprio método `plot()` aceita outro `Waterfall`, permitindo comparar os
dados antes e depois do processamento.


In [ ]:
wf.plot(
    other=wf_pipeline,
    title="Original × processado",
    use_datashader=False,
    include_lightcurve=True,
    include_sed=True,
    width=430,
    height=260,
)


## 13. Resumo

Neste notebook utilizamos a API pública de `radioskysims-waterfall`
para demonstrar o fluxo completo:

**TOD 2D → Waterfall → processamento → SED / LightCurve → visualização**

Principais recursos demonstrados:

- `Waterfall` e `WaterfallMetadata`;
- operações chainable;
- filtros espectral e temporal;
- baseline e normalização;
- `SED`;
- `LightCurve`;
- `WaterfallPipeline`;
- visualização e comparação entre dados originais e processados.

O exemplo é leve, determinístico e independente de banco de dados,
Space-Track ou acesso à rede.
